In [20]:
import warnings, os
warnings.filterwarnings("ignore")
import multiprocessing as mp
from collections import defaultdict
import numpy as np
import pandas as pd

# 1. Make expression matrix

    Read-based
    UMI-based

In [9]:
info = pd.read_csv("../../../1_NanoNASCseq/reports/NanoNASCseq_Summary.csv", index_col=0)

data = dict()
d = info[(info["Cell_Type"] == "K562") & (info["Time"] == 3) & (info["ActD"].isna()) & (info["UMIs"] >= 5000)]
data["K562.s4U_0uM_180min"] = d[d["s4U"] == 0]
data["K562.s4U_50uM_180min"] = d[(d["s4U"] == 50) & (d["TC.Ratio"] >= 0.008)]
d = info[(info["Cell_Type"] == "mESC") & (info["Time"] == 3) & (info["ActD"].isna()) & (info["UMIs"] >= 5000)]
data["mESC.s4U_0uM_180min"] = d[d["s4U"] == 0]
data["mESC.s4U_400uM_180min"] = d[(d["s4U"] == 400) & (d["TC.Ratio"] >= 0.01)]
for name, d in data.items():
    print(d.shape, name, sep="\t")

(190, 48)	K562.s4U_0uM_180min
(105, 48)	K562.s4U_50uM_180min
(64, 48)	mESC.s4U_0uM_180min
(110, 48)	mESC.s4U_400uM_180min


In [10]:
for name, d in data.items():
    categories = ["full-splice_match", "incomplete-splice_match", "novel_in_catalog", "novel_not_in_catalog"]
    array1 = [] # read-based
    array2 = [] # umi-based
    array3 = [] # umi-based (mrUMI only)
    
    for cell in d.index:
        run = cell.split(".")[0]
        path1 = "../../../1_NanoNASCseq/results/5_expression/3_sqanti3/%s/%s/%s_classification.txt" % (run, cell, cell)
        d1 = pd.read_csv(path1, sep="\t", header=0, index_col=0)
        path2 = "../../../1_NanoNASCseq/results/3_mapping/5_mark_duplicate/%s/%s.tsv" % (run, cell)
        d2 = pd.read_csv(path2, sep="\t")
        d2 = d2[["UMI", "AllSize"]]
        d = d1.merge(d2, left_index=True, right_on="UMI")
        
        d = d[[x in categories for x in d["structural_category"]]]
        counter1 = defaultdict(int) # read-based
        counter2 = defaultdict(int) # umi-based
        counter3 = defaultdict(int) # umi-based (mrUMI only)
        for gid, reads in d[["associated_gene", "AllSize"]].values:
            counter1[gid] += reads
            counter2[gid] += 1
            if reads >= 2:
                counter3[gid] += 1
        s1 = pd.Series(counter1, name=cell)
        s2 = pd.Series(counter2, name=cell)
        s3 = pd.Series(counter3, name=cell)
        array1.append(s1)
        array2.append(s2)
        array3.append(s3)
    
    m1 = pd.concat(array1, axis=1).fillna(0) # read-based
    m1.index.name = "GeneID"
    m2 = pd.concat(array2, axis=1).fillna(0) # umi-based
    m2.index.name = "GeneID"
    m3 = pd.concat(array3, axis=1).fillna(0) # umi-based (mrUMI only)
    m3.index.name = "GeneID"
    
    m1.to_csv("results/matrix/%s.read_based.csv" % name)
    m2.to_csv("results/matrix/%s.umi_based.csv" % name)
    m3.to_csv("results/matrix/%s.umi_based.mrUMI.csv" % name)

# Make pseudo-bulk expression

In [13]:
info = pd.read_csv("../../../1_NanoNASCseq/reports/NanoNASCseq_Summary.csv", index_col=0)
data = dict()
d = info[(info["Cell_Type"] == "K562") & (info["Time"] == 3) & (info["ActD"].isna()) & (info["UMIs"] > 5000)]
data["K562.s4U_0uM_180min"] = d[d["s4U"] == 0]
data["K562.s4U_50uM_180min"] = d[(d["s4U"] == 50) & (d["TC.Ratio"] > 0.008)]
d = info[(info["Cell_Type"] == "mESC") & (info["Time"] == 3) & (info["ActD"].isna()) & (info["UMIs"] > 5000)]
data["mESC.s4U_0uM_180min"] = d[d["s4U"] == 0]
data["mESC.s4U_400uM_180min"] = d[(d["s4U"] == 400) & (d["TC.Ratio"] > 0.01)]
for name, d in data.items():
    print(d.shape, name, sep="\t")

(190, 48)	K562.s4U_0uM_180min
(105, 48)	K562.s4U_50uM_180min
(64, 48)	mESC.s4U_0uM_180min
(110, 48)	mESC.s4U_400uM_180min


In [19]:
def make_pseudobulk_expression(cells, time, species, base, tc, prefix):
    path1 = "%s.%dTC.%s_based.filelist.txt" % (prefix, tc, base)
    path2 = "%s.%dTC.%s_based.csv" % (prefix, tc, base)
    path3 = "%s.%dTC.%s_based.annotated.csv" % (prefix, tc, base)

    if base == "gene":
        if species == "Human":
            anno = pd.read_csv("/home/chenzonggui/species/homo_sapiens/GRCh38.p13/gencode.v39.gene_info.csv", index_col=0)
        else:
            anno = pd.read_csv("/home/chenzonggui/species/mus_musculus/GRCm38.p6/gencode.vM25.gene_info.csv", index_col=0)
    else:
        if species == "Human":
            anno = pd.read_csv("/home/chenzonggui/species/homo_sapiens/GRCh38.p13/gencode.v39.transcript_info.csv", index_col=0)
        else:
            anno = pd.read_csv("/home/chenzonggui/species/mus_musculus/GRCm38.p6/gencode.vM25.transcript_info.csv", index_col=0)

    with open(path1, "w+") as fw:
        for cell in cells:
            if base == "gene":
                path = "../../../1_NanoNASCseq/results/5_expression/4_quant_genes/min_read_2_min_tc_%d/%s/%s.tsv" % (tc, cell.split(".")[0], cell)
            else:
                path = "../../../1_NanoNASCseq/results/5_expression/6_quant_isoforms/min_read_2_min_tc_%d/%s/%s.tsv" % (tc, cell.split(".")[0], cell)
            fw.write(path + "\n")     
    
    ! ../../../1_NanoNASCseq/scripts/expression/merge_counts.py {path1} {path2}
    
    d = pd.read_csv(path2, index_col=0)
    d["TPM"] = d["Total"] * 1e6 / d["Total"].sum()
    d["TP10K"] = d["Total"] * 1e4 / d["Total"].sum()
    d["NTR"] = d["New"] / d["Total"]
    d["Halflife"] = np.nan if t == 0 else -t / np.log2(1 - d["NTR"])
    d["DecayRate"] = np.log(2) / d["Halflife"]
    d["SynthesisRate"] = d["DecayRate"] * d["TP10K"]
    d = d.merge(anno, left_index=True, right_index=True)
    d.to_csv(path3)
    os.remove(path2)


pool = mp.Pool(24)
for name, info2 in data.items():
    for base in ["gene", "transcript"]:
        for tc in [1, 2]:
            time = 3 if "180min" in name else 0
            species = "Human" if "K562" in name else "Mouse"
            prefix = "results/pseudobulk/%s" % name
            args = (info2.index, time, species, base, tc, prefix)
            pool.apply_async(make_pseudobulk_expression, args)
pool.close()
pool.join()